# Pipeline: PMD calling & insulation

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}clustering/merged/group_meta.tsv'`  ·  _metadata_
- `f'{indir}merged_allc/cluster_donor.mcds'`  ·  _sc/pseudobulk mC (allc)_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [2]:
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from concurrent.futures import ProcessPoolExecutor, as_completed

import pysam
import anndata

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [3]:
indir = f'{ENTEX_ROOT}/'


In [4]:
group_meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', sep='\t', header=0, index_col=0)
# group_meta = group_meta[['L2_any', 'L1', 'count']]
group_meta['L1_annot'] = group_meta['L1_annot'].str.replace(' ','-').str.replace('/','_')
annot2L1 = group_meta[['L1','L1_annot']].set_index('L1_annot')['L1'].to_dict()
L1annot = group_meta[['L1','L1_annot']].set_index('L1')['L1_annot'].to_dict()


In [5]:
from ALLCools.mcds import MCDS

mcds = MCDS.open(f'{indir}merged_allc/cluster_donor.mcds', var_dim='chrom1k')
mcds

In [6]:
mcds = mcds.assign_coords(L1=('cell', group_meta.loc[mcds.get_index('cell'), 'L1']))
mcds = mcds.groupby('L1').sum()


In [7]:
# mcds = mcds.assign_coords(chrom10k=('chrom1k', bin_df['chrom1k']))
mcds = mcds.sel({'mc_type':'CGN'})
mcds = MCDS(mcds, obs_dim='L1', var_dim='chrom1k')
mcds

In [8]:
exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [9]:
mcds['chrom1k_da_frac'] = mcds['chrom1k_da'].sel({'count_type':'mc'}) / mcds['chrom1k_da'].sel({'count_type':'cov'})
# .add_mc_frac(normalize_per_cell=False)


In [10]:
bin_df = mcds[['chrom1k_chrom', 'chrom1k_start', 'chrom1k_end']].to_pandas()


In [11]:
bin_df = bin_df[['chrom1k_chrom','chrom1k_start','chrom1k_end']]
bin_df.columns = ['chrom', 'start', 'end']

In [12]:
data = mcds['chrom1k_da_frac'].to_pandas().fillna(0)
print(data.shape)

In [13]:
from pybedtools import BedTool

black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
alt_allele_path = f'{REF_ROOT}/hg38/fasta/hg38.altseq.bed'

def remove_region(black_list_path):
    feature_bed = BedTool.from_dataframe(bin_df)
    black_list_bed = BedTool(black_list_path)
    black_feature = feature_bed.intersect(black_list_bed, f=0.2, wa=True)
    black_feature_index = black_feature.to_dataframe().set_index(["chrom", "start", "end"]).index
    black_feature_id = pd.Index(
        bin_df.reset_index()
        .set_index(["chrom", "start", "end"])
        .loc[black_feature_index][bin_df.index.name]
    )
    print(black_feature_id.shape[0])
    return black_feature_id


In [14]:
black_feature_id = remove_region(black_list_path)
alt_feature_id = remove_region(alt_allele_path)


In [15]:
data.loc[:, data.columns.isin(black_feature_id) | data.columns.isin(alt_feature_id)] = 0

In [16]:
import math
from scipy.stats import ranksums
ws = 25
res = 1000
vmin = 0.05
pvmax = 1

def callpmd_worker(chrom, ct):
    chrfilter = bin_df['chrom'] == chrom
    datachr = data.loc[ct, chrfilter].values[np.argsort(bin_df.loc[chrfilter, 'start'])].copy()
    conv = np.convolve(datachr, np.ones(ws), mode='valid')
    diff = np.abs(conv[:-ws] - conv[ws:])
    print(diff.shape[0], datachr.shape[0])
    s, t = (0, diff.shape[0])
    turn = [s]
    while s < t - 2:
        last = math.hypot(1, diff[s + 1] - diff[s])
        for k in range(s + 2, t):
            slope = (diff[k] - diff[s]) / (k - s)
            l = np.abs([diff[i] - diff[s] - slope * (i - s) for i in range(s + 1, k)]).sum()
            tmp = math.hypot(k - s, diff[k] - diff[s]) - l
            if tmp < last:
                s = k - 1
                last = math.hypot(1, diff[s + 1] - diff[s])
                turn.append(s)
                break
            else:
                last = tmp
        if k >= t - 1:
            s = k
            turn.append(s)
    turn = np.array(turn)
    pv = np.array([True] + [ranksums(datachr[xx:xx + ws], datachr[xx + ws:xx + 2 * ws])[1] for xx in turn[1:-1]] + [True])
    localmax_filter = np.logical_and(diff[turn[1:-1]] > diff[turn[:-2]], diff[turn[1:-1]] > diff[turn[2:]])
    localmax_filter = [True] + localmax_filter.tolist() + [True]
    localmax = turn[np.logical_and(pv < pvmax, localmax_filter)]
    print(localmax.shape)
    cumsum = np.cumsum(datachr)
    segave = (cumsum[localmax[1:] + ws] - cumsum[localmax[:-1] + ws]) / (localmax[1:] - localmax[:-1])
    selseg = np.logical_and(segave < vmax, segave > vmin)
    pmdstart = localmax[1:-1][np.logical_and(~selseg[:-1], selseg[1:])]
    if selseg[0]:
        pmdstart = [localmax[0]] + list(pmdstart)
    pmdend = localmax[1:-1][np.logical_and(selseg[:-1], ~selseg[1:])]
    if selseg[-1]:
        pmdend = list(pmdend) + [localmax[-1]]
    pmdchr = pd.DataFrame(np.array([pmdstart, pmdend]), index=['start', 'end']).T + ws
    pmdchr['ave'] = (cumsum[pmdchr['end']] - cumsum[pmdchr['start']]) / (pmdchr['end'] - pmdchr['start'])
    pmdchr[['start', 'end']] = pmdchr[['start', 'end']] * res
    pmdchr['chrom'] = chrom
    return (datachr, diff, turn, localmax, pmdchr)

In [17]:
datatmp = data.loc[:, ~(data.columns.isin(black_feature_id) | data.columns.isin(alt_feature_id))]
cgmean = datatmp.mean(axis=1)
cgmedian = datatmp.median(axis=1)

# cgmean = np.abs(datatmp - 0.5).mean(axis=1) + 0.5
# cgmedian = np.abs(datatmp - 0.5).median(axis=1) + 0.5

In [18]:
ct = 'c2'
tmp = np.random.choice(data.loc[ct].values, 100000, False)

In [19]:
ct = 'c0'
tmp = np.random.choice(data.loc[ct].values, 100000, False)

In [20]:
ct = 'c7'
tmp = np.random.choice(data.loc[ct].values, 100000, False)

In [21]:
ct = 'c22'
tmp = np.random.choice(data.loc[ct].values, 100000, False)

In [22]:
from concurrent.futures import ProcessPoolExecutor, as_completed

ct = 'c0'
vmax = cgmedian.loc[ct]
cpu = 32
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for chrom in bin_df['chrom'].unique():
        future = executor.submit(
            callpmd_worker,
            chrom=chrom,
            ct=ct,
        )
        futures[future] = chrom

    pmd = []
    for future in as_completed(futures):
        chrom = futures[future]
        datachr, diff, turn, localmax, pmdchr = future.result()
        pmd.append(pmdchr)
        print(f'{chrom} finished')
        

In [23]:
vmax

In [24]:
pmd = pd.concat(pmd, axis=0)[['chrom', 'start', 'end', 'ave']]
pmd.to_csv(f'pmd_calling/{ct}_pmd_ws{ws}_nopv_median.bed', sep='\t', index=False, header=False)


In [25]:
chrom, start, end = 'chr1', 7000000, 8500000
res = 1000


In [26]:
ct = 'c0'
vmax = cgmean.loc[ct]
datachr, diff, turn, localmax, pmdchr = callpmd_worker(ct=ct, chrom='chr1')


In [27]:
localmax_filter = np.logical_and(diff[turn[1:-1]] > diff[turn[:-2]], diff[turn[1:-1]] > diff[turn[2:]])
localmax_filter = [True] + localmax_filter.tolist() + [True]
localmax = turn[localmax_filter]
print(localmax.shape)
cumsum = np.cumsum(datachr)
segave = (cumsum[localmax[1:] + ws] - cumsum[localmax[:-1] + ws]) / (localmax[1:] - localmax[:-1])
selseg = np.logical_and(segave < vmax, segave > vmin)
pmdstart = localmax[1:-1][np.logical_and(~selseg[:-1], selseg[1:])]
if selseg[0]:
    pmdstart = [localmax[0]] + list(pmdstart)
pmdend = localmax[1:-1][np.logical_and(selseg[:-1], ~selseg[1:])]
if selseg[-1]:
    pmdend = list(pmdend) + [localmax[-1]]
pmdchr = pd.DataFrame(np.array([pmdstart, pmdend]), index=['start', 'end']).T + ws
pmdchr['ave'] = (cumsum[pmdchr['end']] - cumsum[pmdchr['start']]) / (pmdchr['end'] - pmdchr['start'])
pmdchr[['start', 'end']] = pmdchr[['start', 'end']] * res
pmdchr['chrom'] = chrom

In [28]:
tmp = datachr[start // res:end // res]
x = np.arange(tmp.shape[0])
pmdtmp = (pmdchr.loc[(pmdchr['end'] > start) & (pmdchr['start'] < end), ['start', 'end']].values - start) // res
tmp = np.zeros(datachr.shape[0])
tmp[ws:-ws + 1] = np.abs(diff)
tmp = tmp[start // res:end // res]
x = np.arange(tmp.shape[0])
regionfilter = np.logical_and(turn + ws >= start // res, turn + ws < end // res)
turntmp = turn[regionfilter]
regionfilter = np.logical_and(localmax + ws >= start // res, localmax + ws < end // res)
tmp = localmax[regionfilter]

In [29]:
tmp = datachr[start // res:end // res]
x = np.arange(tmp.shape[0])
pmdtmp = (pmdchr.loc[(pmdchr['end'] > start) & (pmdchr['start'] < end), ['start', 'end']].values - start) // res
tmp = np.zeros(datachr.shape[0])
tmp[ws:-ws + 1] = np.abs(diff)
tmp = tmp[start // res:end // res]
x = np.arange(tmp.shape[0])
regionfilter = np.logical_and(turn + ws >= start // res, turn + ws < end // res)
turntmp = turn[regionfilter]
regionfilter = np.logical_and(localmax + ws >= start // res, localmax + ws < end // res)
tmp = localmax[regionfilter]

In [30]:
ymin = 0.6
tmp = datachr[start // res:end // res]
x = np.arange(tmp.shape[0])
pmdtmp = (pmdchr.loc[(pmdchr['end'] > start) & (pmdchr['start'] < end), ['start', 'end']].values - start) // res
tmp = np.zeros(datachr.shape[0])
tmp[ws:-ws + 1] = np.abs(diff)
tmp = tmp[start // res:end // res]
x = np.arange(tmp.shape[0])
regionfilter = np.logical_and(turn + ws >= start // res, turn + ws < end // res)
turntmp = turn[regionfilter]
regionfilter = np.logical_and(localmax + ws >= start // res, localmax + ws < end // res)
tmp = localmax[regionfilter]

In [31]:
ymin = 0.6
tmp = datachr[start // res:end // res]
x = np.arange(tmp.shape[0])
pmdtmp = (pmdchr.loc[(pmdchr['end'] > start) & (pmdchr['start'] < end), ['start', 'end']].values - start) // res
tmp = np.zeros(datachr.shape[0])
tmp[ws:-ws + 1] = np.abs(diff)
tmp = tmp[start // res:end // res]
x = np.arange(tmp.shape[0])
regionfilter = np.logical_and(turn + ws >= start // res, turn + ws < end // res)
turntmp = turn[regionfilter]
regionfilter = np.logical_and(localmax + ws >= start // res, localmax + ws < end // res)
tmp = localmax[regionfilter]

In [36]:
chrfilter = (bin_df['chrom']==chrom)
datachr = data.loc['c2', chrfilter].values[np.argsort(bin_df.loc[chrfilter, 'start'])]
datachr.shape ##chrlen

In [37]:
ws = 25
conv = np.convolve(datachr, np.ones(ws), mode='valid') ##chrlen-ws+1; i is average of datachr[i:(i+ws)]
diff = np.abs(conv[:-ws] - conv[ws:]) ##chrlen-2ws+1; 0 -> datachr[ws], i is diff between datachr[i:(i+ws)] and datachr[(i+ws):(i+2ws)]
print(diff.shape[0], datachr.shape[0])


In [38]:
import math

s, t = 0, diff.shape[0]
turn = [s]
while s<(t-2):
    last = math.hypot(1, diff[s+1]-diff[s]) ## initialize with 2 points
    for k in range(s+2,t):
        slope = (diff[k] - diff[s]) / (k - s)
        l = np.abs([diff[i] - diff[s] - slope*(i-s) for i in range(s+1, k)]).sum()
        tmp = math.hypot(k-s, diff[k]-diff[s]) - l
        if (tmp<last):
            s = k - 1
            last = math.hypot(1, diff[s+1]-diff[s])
            turn.append(s)
            break
        else:
            last = tmp
    if k>=(t-1):
        s = k
        turn.append(s)
        
turn = np.array(turn) ##position in diff, 0 -> datachr[ws]


In [39]:
from scipy.stats import ranksums

pv = np.array([True] + [ranksums(datachr[xx:(xx+ws)], datachr[(xx+ws):(xx+2*ws)])[1] for xx in turn[1:-1]] + [True]) 
##shape n_turn, position in diff


In [40]:
# from sklearn.metrics import roc_auc_score
# label = np.repeat([1, 0], ws)
# roc = np.array([roc_auc_score(label, datachr[xx:(xx+2*ws)]) for xx in turn])
# roc = np.abs(roc - 0.5) + 0.5


In [41]:
# print((pv<0.05).sum(), (roc>0.8).sum())


In [42]:
localmax_filter = np.logical_and(diff[turn[1:-1]]>diff[turn[:-2]], diff[turn[1:-1]]>diff[turn[2:]])
localmax_filter = [True] + localmax_filter.tolist() + [True]
## shape n_turn

In [43]:
localmax = turn[np.logical_and(pv < 0.05, localmax_filter)]
print(localmax.shape)

In [44]:
cumsum = np.cumsum(datachr)
segave = (cumsum[localmax[1:]+ws] - cumsum[localmax[:-1]+ws]) / (localmax[1:] - localmax[:-1]) ## shape n_localmax-1, i is ave mcg between localmax[i] and localmax[i+1]
selseg = np.logical_and(segave<0.6, segave>0.05)
pmdstart = localmax[1:-1][np.logical_and(~selseg[:-1], selseg[1:])] ## index in localmax
if selseg[0]:
    pmdstart = [localmax[0]] + list(pmdstart)
pmdend = localmax[1:-1][np.logical_and(selseg[:-1], ~selseg[1:])] ## index in localmax
if selseg[-1]:
    pmdend = list(pmdend) + [localmax[-1]]
pmdchr = (pd.DataFrame(np.array([pmdstart, pmdend]), index=['start', 'end']).T) + ws
pmdchr['ave'] = (cumsum[pmdchr['end']] - cumsum[pmdchr['start']]) / (pmdchr['end'] - pmdchr['start'])
pmdchr[['start', 'end']] = pmdchr[['start', 'end']] * res
pmdchr['chrom'] = chrom


In [45]:
# regionfilter = np.logical_and((localmax+ws)>=start//res, (localmax+ws)<end//res) ## shape n_turn
# tmp = localmax[regionfilter]
# print(tmp.shape)

In [46]:
# segave[np.logical_and((localmax[1:]+ws)<(end//res), (localmax[:-1]+ws)>(start//res))].shape

In [47]:
# segave[np.logical_and((localmax[1:]+ws)<(end//res), (localmax[:-1]+ws)>(start//res))]